In [ ]:
#To load automatically updates from libraries without re-running import cells
%load_ext autoreload
%autoreload 2

### Preparing training and test data

In [ ]:
from astropy.io import fits
from astropy.table import Table
from pre_processing_utils import (
    remove_corrupted_rows,
    get_puls_and_AGN_dataframe,
    scale_df_data,
    get_network_data_from_df_data,
)

catalog_4FGL = fits.open("4FGL-DR2.fit")
fits_data = catalog_4FGL[1].data

catalog = Table(fits_data)

catalog = remove_corrupted_rows(catalog)  # Remove catalog rows with missing data

df = get_puls_and_AGN_dataframe(catalog)

scaled_df = scale_df_data(df)
# scaled_df = add_category_id(scaled_df)

random_state = 52
X_train, X_val, X_test, y_train, y_val, y_test, class_weights_dict = (
    get_network_data_from_df_data(scaled_df, random_state)
)

### Parameters scan

In [ ]:
from sequential_network import *

layers = [1, 2, 3]
neurons = [32, 64, 128]
# batch_sizes = [16, 32, 64, 128]
batch_sizes = [16]

netw_opt_params = Sequential_Network.scan_network_parameters(
    X_train,
    y_train,
    X_val,
    y_val,
    len(X_train[0]),
    layers,
    neurons,
    batch_sizes,
    2,
    class_weight=class_weights_dict,
)

print(netw_opt_params)

### Re-train on optimal parameter

In [ ]:
seq_netw = Sequential_Network(
    len(X_train[0]), 3, 32, 2, class_weight=class_weights_dict
)
seq_netw.build_model()
seq_netw.compile_model()
train_history = seq_netw.train(
    x_train=X_train,
    y_train=y_train,
    x_val=X_val,
    y_val=y_val,
    batch_size=16,
    epochs=200,
)

model_loss = sum(train_history.history["val_loss"][-3:]) / 3

### Result visualization

In [ ]:
from constants import SOURCES_CATEGORIES

# score=model.evaluate(X_test,Y_test,verbose=verbose)
conf_matrix = seq_netw.make_prediction(X_test, y_test, SOURCES_CATEGORIES)
print(conf_matrix)

seq_netw.get_performance_metrics(X_test, y_test)

In [ ]:
display(scaled_df.head(5))
print(y_train)